# Fake News Detection using Natural Language Processing (NLP)

**IBM Naan Mudhalvan AI101** \
*Anna University (Madras Institute of Technology)*

---

### TEAM DETAILS:
- **VIJAI SURIA M** (Email: vijaisuria04@gmail.com | NM ID: CE95F21BA7E422D8E5F4293703737B6D)
- **NITHISH T** (Email: hariveena9787@gmail.com | NM ID: EDFD0990FC19F8C87E541276D37DEE6A)
- **KIRAN KUMAR M** (Email: kumarkiran0893@gmail.com | NM ID: 048ECB0D9F48C56BF304011FE331B325)
- **GIRIDHARAN S S** (Email: girirohitlic777@gmail.com | NM ID: 60517DC79B0227181F3441F69DE5CFF9)

---

## 📌 Project Overview
Fake news propagation is a significant challenge in modern media. In this project, we construct an end-to-end Machine Learning pipeline using Natural Language Processing (NLP) to classify news articles as **Real** or **Fake**. 

### ⚙️ Workflow Steps:
1. **Step 1: Import Necessary Libraries**
2. **Step 2: Load and Explore the Dataset**
3. **Step 3: Data Preprocessing & Cleaning**
4. **Step 4: Feature Extraction (TF-IDF Vectorization)**
5. **Step 5: Split Data into Training and Testing Sets**
6. **Step 6: Model Training & Evaluation** (Multinomial Naive Bayes, Decision Tree, Passive Aggressive, Random Forest, Logistic Regression)
7. **Step 7: Model Comparison & Validation**
8. **Step 8: Interactive Prediction Pipeline for Custom News**


## Step 1: Import Necessary Libraries


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import string
import os

# Scikit-Learn Modules
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import PassiveAggressiveClassifier, LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, confusion_matrix, 
                             classification_report, precision_score, 
                             recall_score, f1_score)

# NLTK Setup
import nltk
from nltk.corpus import stopwords

# Download required NLTK resources safely
for resource in ['punkt', 'punkt_tab', 'stopwords']:
    try:
        nltk.download(resource, quiet=True)
    except Exception as e:
        print(f"Notice downloading {resource}: {e}")

print("✅ All required libraries imported successfully!")


## Step 2: Load and Explore the Dataset


In [ ]:
# Robust Dataset Path Loading (Works locally and in Google Colab)
true_path = 'dataset/True.csv' if os.path.exists('dataset/True.csv') else '/content/drive/MyDrive/Colab Notebooks/fake-news-detective/dataset/True.csv'
fake_path = 'dataset/Fake.csv' if os.path.exists('dataset/Fake.csv') else '/content/drive/MyDrive/Colab Notebooks/fake-news-detective/dataset/Fake.csv'

true_df = pd.read_csv(true_path)
false_df = pd.read_csv(fake_path)

# Labeling: 1 for Real News, 0 for Fake News
true_df['label'] = 1
false_df['label'] = 0

print(f"Real News count: {len(true_df)}")
print(f"Fake News count: {len(false_df)}")

# Combine both datasets and shuffle
data = pd.concat([true_df, false_df], axis=0).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Combined Dataset Shape: {data.shape}")
print("\nMissing Values Check:")
print(data.isnull().sum())

# Display sample records
data.head()


In [ ]:
# Visualize Target Class Distribution
plt.figure(figsize=(6, 4))
sns.countplot(x='label', data=data, palette=['#e74c3c', '#2ecc71'])
plt.xticks([0, 1], ['Fake News (0)', 'Real News (1)'])
plt.title('Distribution of Real vs Fake News Articles', fontsize=12, fontweight='bold')
plt.xlabel('Category')
plt.ylabel('Count')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()


## Step 3: Data Preprocessing & Cleaning

In this step, we perform text cleaning:
1. Handle missing values.
2. Combine `title` and `text` into a single feature for comprehensive context.
3. Convert text to lower case.
4. Remove URLs, HTML tags, punctuation, special characters, and digits.
5. Filter out English stop words using NLTK.


In [ ]:
# Drop rows with missing text content
data = data.dropna(subset=['text', 'title']).copy()

# Combine title and text for rich context
data['full_content'] = data['title'].fillna('') + " " + data['text'].fillna('')

# Define stop words set
stop_words = set(stopwords.words('english'))

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'https?://\S+|www\.\S+', '', text) # Remove URLs
    text = re.sub(r'<.*?>+', '', text)                  # Remove HTML tags
    text = re.sub(r'[%s]' % re.escape(string.punctuation), '', text) # Remove punctuation
    text = re.sub(r'\n', ' ', text)                    # Remove newlines
    text = re.sub(r'\w*\d\w*', '', text)             # Remove words with numbers
    words = text.split()
    words = [w for w in words if w not in stop_words and len(w) > 2]
    return " ".join(words)

print("Cleaning text data... (This may take a few seconds)")
data['cleaned_content'] = data['full_content'].apply(clean_text)

print("✅ Text Preprocessing Completed Successfully!")
print("\nSample Original Content:")
print(data['full_content'].iloc[0][:200] + "...")
print("\nSample Cleaned Content:")
print(data['cleaned_content'].iloc[0][:200] + "...")


## Step 4: Feature Extraction (TF-IDF Vectorization)

We transform cleaned text articles into numerical features using **TF-IDF (Term Frequency - Inverse Document Frequency)** vectorization with a limit of 5,000 top features and unigram/bigram ranges.


In [ ]:
# Initialize TF-IDF Vectorizer
tfidf_vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))

# Fit and transform the cleaned text
X = tfidf_vectorizer.fit_transform(data['cleaned_content'])
y = data['label'].values

print(f"TF-IDF Feature Matrix Shape: {X.shape}")
print(f"Target Array Shape: {y.shape}")


## Step 5: Split the Data into Training and Testing Sets


In [ ]:
# Split dataset into 80% Training and 20% Testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape:  {y_test.shape}")


## Step 6: Model Training & Evaluation

We train and evaluate 5 different Machine Learning classification models:
1. **Multinomial Naive Bayes**
2. **Decision Tree Classifier**
3. **Passive Aggressive Classifier**
4. **Random Forest Classifier**
5. **Logistic Regression**


In [ ]:
# Helper dictionary to store all model results
model_results = {}

def evaluate_model(name, model, X_tr, y_tr, X_te, y_te):
    print(f"==================== Training {name} ====================")
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    
    acc = accuracy_score(y_te, y_pred)
    prec = precision_score(y_te, y_pred)
    rec = recall_score(y_te, y_pred)
    f1 = f1_score(y_te, y_pred)
    cm = confusion_matrix(y_te, y_pred)
    
    model_results[name] = {
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1,
        'Model': model
    }
    
    print(f"🎯 {name} Accuracy: {acc:.4f} ({acc*100:.2f}%)")
    print(f"Classification Report:\n{classification_report(y_te, y_pred, target_names=['Fake', 'Real'])}")
    
    # Plot Confusion Matrix
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['Fake (0)', 'Real (1)'], 
                yticklabels=['Fake (0)', 'Real (1)'])
    plt.title(f'Confusion Matrix: {name}', fontsize=11, fontweight='bold')
    plt.xlabel('Predicted Label')
    plt.ylabel('Actual Label')
    plt.tight_layout()
    plt.show()
    
    return model


### 1. Multinomial Naive Bayes Model


In [ ]:
mnb_model = evaluate_model('Multinomial Naive Bayes', MultinomialNB(), X_train, y_train, X_test, y_test)


### 2. Decision Tree Classifier


In [ ]:
dt_model = evaluate_model('Decision Tree', DecisionTreeClassifier(random_state=42), X_train, y_train, X_test, y_test)


### 3. Passive Aggressive Classifier


In [ ]:
pa_model = evaluate_model('Passive Aggressive Classifier', PassiveAggressiveClassifier(max_iter=50, random_state=42), X_train, y_train, X_test, y_test)


### 4. Random Forest Classifier


In [ ]:
rf_model = evaluate_model('Random Forest', RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1), X_train, y_train, X_test, y_test)


### 5. Logistic Regression Model


In [ ]:
lr_model = evaluate_model('Logistic Regression', LogisticRegression(max_iter=1000, random_state=42), X_train, y_train, X_test, y_test)


## Step 7: Model Comparison & Performance Validation

We summarize and visualize the performance of all 5 classifiers to select the best model for deployment.


In [ ]:
# Create Performance Comparison Table
results_df = pd.DataFrame(model_results).T[['Accuracy', 'Precision', 'Recall', 'F1-Score']]
results_df = results_df.sort_values(by='Accuracy', ascending=False)

print("🏆 Model Performance Comparison Summary:")
print(results_df)

# Visual Comparison Plot
plt.figure(figsize=(10, 5))
ax = results_df.plot(kind='bar', figsize=(11, 5), colormap='viridis', edgecolor='black')
plt.title('Machine Learning Models Performance Metrics Comparison', fontsize=13, fontweight='bold')
plt.ylabel('Score')
plt.xlabel('Algorithm')
plt.ylim(0.85, 1.01)
plt.xticks(rotation=15, ha='right')
plt.legend(loc='lower right')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

best_model_name = results_df.index[0]
best_accuracy = results_df.iloc[0]['Accuracy']
print(f"⭐ Top Performing Model: {best_model_name} with Accuracy of {best_accuracy*100:.2f}%")


## Step 8: Interactive Prediction Pipeline

We test our top model with brand-new, custom news articles to evaluate real-time fake news detection capabilities.


In [ ]:
best_model = model_results[best_model_name]['Model']

def predict_news(news_text):
    cleaned = clean_text(news_text)
    vectorized = tfidf_vectorizer.transform([cleaned])
    prediction = best_model.predict(vectorized)[0]
    
    confidence = ""
    if hasattr(best_model, "predict_proba"):
        probs = best_model.predict_proba(vectorized)[0]
        confidence = f" (Confidence: {max(probs)*100:.2f}%)"
    
    label = "🟢 REAL NEWS" if prediction == 1 else "🔴 FAKE NEWS"
    print(f"Prediction: {label}{confidence}")
    print(f"Article Excerpt: {news_text[:140]}...\n")

# Demo Predictions
print("--- Interactive Prediction Test ---\n")

sample_real = "WASHINGTON (Reuters) - The U.S. Senate passed the bipartisan infrastructure package with a 69-30 vote on Tuesday."
sample_fake = "BREAKING: Secret lunar base discovered hosting hidden summit with top world leaders!"

print("Test 1 (Real News Sample):")
predict_news(sample_real)

print("Test 2 (Fake News Sample):")
predict_news(sample_fake)
